# 대표 장르 추출 및 층화 작업

## 목적
Steam 인디 게임 9,692개 중 리뷰 감성 분석을 위한 샘플링 대상 게임을 선별하기 위해, 각 게임의 대표 장르를 추출한다.

## 배경
- Steam `appdetails` API의 `genres` 필드는 알파벳 순으로 정렬된 멀티레이블 리스트로, "주 장르" 개념이 없음
- 층화 샘플링을 위해 게임당 하나의 대표 장르 배정이 필요
- 대표 장르 선정 방식: **희귀 장르 우선** — 게임이 가진 장르 중 전체 데이터에서 등장 빈도가 가장 낮은 장르를 대표로 선정

## 한계
- 희귀 장르가 반드시 해당 게임의 핵심 장르를 의미하지는 않음
- **Adventure · Casual 버킷 대표성 주의**: 두 장르는 Steam에서 범용적으로 붙는 태그라 원본 게임 수 대비 대표 장르 배정 수가 크게 줄어든다 (Adventure 84% 감소, Casual 74% 감소). 해당 버킷은 '다른 장르가 없는 순수 Adventure/Casual'에 가까우며, Adventure/Casual 전체를 대표하지 않으므로 장르 간 비교 시 해석에 주의가 필요하다.
- Sports/Racing 버킷에는 해당 장르가 부수적인 게임이 포함될 수 있음
- Steam 데이터 특성상 완전한 대표 장르 선정은 불가능하며, 이 한계를 감안하여 해석 필요

## 라이브러리 임포트

In [1]:
import ast
import warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

pd.set_option('display.max_columns', None)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

print('라이브러리 로드 완료')

라이브러리 로드 완료


## 데이터 로드 및 파생 컬럼 생성

`steam_indie_games.csv`를 불러오고 분석에 필요한 파생 컬럼을 추가합니다.

- `total_reviews`: 긍정 + 부정 리뷰 합산
- `positive_rate`: 긍정 리뷰 비율 (%)
- `genres`: 문자열 리스트 파싱

In [2]:
df = pd.read_csv('../../data/preprocessed/steam_indie_games.csv')
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')
df['positive_rate'] = df['positive'] / df['total_reviews'] * 100

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df['genres'] = df['genres'].apply(parse_genres)

print(f'모집단: {len(df):,}개')
print(f'출시연도 분포:')
print(df['release_date'].dt.year.value_counts().sort_index().to_string())

모집단: 8,730개
출시연도 분포:
release_date
2023    3032
2024    3818
2025    1880


## 1. 전체 장르 현황 탐색

분석 대상 장르를 결정하기 위해 전체 장르의 게임 수 분포를 확인합니다.

In [3]:
genre_counts_raw = {}
for genres in df['genres']:
    for genre in genres:
        genre_counts_raw[genre] = genre_counts_raw.get(genre, 0) + 1

genre_df = pd.DataFrame(list(genre_counts_raw.items()), columns=['Genre', 'Count']).sort_values('Count', ascending=False)
display(genre_df)

,Genre,Count
2,Indie,8725
0,Adventure,4496
1,Casual,3870
5,Action,3846
6,Simulation,2308
3,RPG,1967
4,Strategy,1916
8,Sports,323
7,Racing,283


## 2. 대상 장르별 게임 수 분포

8개 대상 장르(`TARGET_GENRES`) 각각에 몇 개의 게임이 속하는지 확인한다.
한 게임이 여러 장르를 가질 수 있으므로 장르별 집계는 중복을 허용한다.

In [4]:
TARGET_GENRES = {'Adventure', 'Casual', 'Action', 'Simulation', 'RPG', 'Strategy', 'Sports', 'Racing'}

# 대상 장르만 필터링하여 게임 수 집계
target_genre_counts = {
    genre: sum(1 for genres in df['genres'] if genre in genres)
    for genre in TARGET_GENRES
}
target_genre_df = (
    pd.DataFrame(list(target_genre_counts.items()), columns=['genre', 'count'])
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

fig = px.bar(
    target_genre_df,
    x='genre',
    y='count',
    text='count',
    title='대상 장르별 게임 수 분포 (중복 허용)',
    labels={'genre': '장르', 'count': '게임 수'},
    color='count',
    color_continuous_scale='Blues',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_categoryorder='total descending',
    coloraxis_showscale=False,
    height=450,
)
fig.show()

print(target_genre_df.to_string(index=False))

     genre  count
 Adventure   4496
    Casual   3870
    Action   3846
Simulation   2308
       RPG   1967
  Strategy   1916
    Sports    323
    Racing    283


## 3. 대표 장르 선정 및 대표 장르 추출

**대표 장르 선정 방식: 희귀 장르 우선** — 게임이 가진 장르 중 전체 데이터에서 등장 빈도가 가장 낮은 장르를 대표로 선정

| 최종 대상 장르 |
|----------|
| Action, Adventure, Casual, Simulation, RPG, Strategy, Sports, Racing |

In [5]:
TARGET_GENRES = {'Adventure', 'Casual', 'Action', 'Simulation', 'RPG', 'Strategy', 'Sports', 'Racing'}

# 00 노트북에서 부적합 장르 및 Indie 단독 게임이 이미 제외됨
# Indie는 모든 게임에 공통으로 붙어 희귀도 기준으로 의미가 없으므로 제외
df['genres_filtered'] = df['genres'].apply(lambda g: [x for x in g if x != 'Indie'])
df_filtered = df[df['genres_filtered'].map(len) > 0].copy()

# 희귀도 계산
genre_count = Counter(genre for genres in df_filtered['genres_filtered'] for genre in genres)
print('=== 장르별 게임 수 (희귀도 기준) ===')
for genre, count in sorted(genre_count.items(), key=lambda x: x[1]):
    print(f'  {genre:25s}: {count:,}')

# 희귀 장르 우선으로 대표 장르 선정
df_filtered['primary_genre'] = df_filtered['genres_filtered'].apply(
    lambda genres: min(genres, key=lambda g: genre_count[g])
)

print('\n=== 대표 장르 분포 ===')
print(df_filtered['primary_genre'].value_counts().to_string())

=== 장르별 게임 수 (희귀도 기준) ===
  Racing                   : 283
  Sports                   : 323
  Strategy                 : 1,916
  RPG                      : 1,967
  Simulation               : 2,308
  Action                   : 3,846
  Casual                   : 3,870
  Adventure                : 4,496

=== 대표 장르 분포 ===
primary_genre
Action        2184
Strategy      1831
RPG           1355
Simulation    1090
Casual        1033
Adventure      721
Racing         283
Sports         233


## 4. 결과 확인

게임별 필터링된 장르 목록과 선정된 대표 장르를 확인한다.

In [6]:
df_filtered[['appid', 'name', 'genres_filtered', 'primary_genre']].head(20)

,appid,name,genres_filtered,primary_genre
0,226620,Desktop Dungeons,"[Adventure, Casual, RPG, Strategy]",Strategy
1,251570,7 Days to Die,"[Action, Adventure, RPG, Simulation, Strategy]",Strategy
2,252190,Defender's Quest 2: Mists of Ruin,"[RPG, Strategy]",Strategy
3,269770,Secrets of Grindea,"[Action, Adventure, RPG]",RPG
4,276870,Dwelvers,"[Simulation, Strategy]",Strategy
5,282880,FaeVerse Alchemy,"[Casual, Strategy]",Strategy
6,290100,Bulwark: Falconeer Chronicles,"[Casual, Simulation, Strategy]",Strategy
7,301280,Skin Deep,"[Action, Simulation]",Simulation
8,318840,Tempopo,[Casual],Casual
9,324470,SinaRun,[Racing],Racing


In [7]:
genre_dist = df_filtered['primary_genre'].value_counts().reset_index()
genre_dist.columns = ['genre', 'count']

fig = px.bar(
    genre_dist,
    x='genre',
    y='count',
    text='count',
    title='대표 장르별 게임 수 분포',
    labels={'genre': '장르', 'count': '게임 수'},
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_categoryorder='total descending')
fig.show()

## 5. 층화 추출

대표 장르(`primary_genre`) × 리뷰 신뢰도(`trust`) 두 축으로 층을 구성한다.

### 층화 변수 기준

| 축 | 층 | 기준 |
|---|---|---|
| 장르 | Action, Adventure, Casual, Simulation, RPG, Strategy, Sports, Racing | `primary_genre` |
| 리뷰 신뢰도 | high | `total_reviews` ≥ 381 (긍정률 오차 ±5% 이하) |
| 리뷰 신뢰도 | mid  | 39 ≤ `total_reviews` < 381 (±5~15%) |
| 리뷰 신뢰도 | low  | `total_reviews` < 39 (±15% 초과) |

리뷰 신뢰도 경계값은 Wilson Score 95% 신뢰 구간 최대 오차 기준으로 산출한다.

### 5-1. Wilson Score 경계값 계산

긍정률 95% CI 최대 오차(worst case: p=0.5)를 기준으로 두 경계값을 산출한다.

- `LOW_BOUNDARY` : 오차 ±15% 이하가 되는 최소 리뷰 수 → low/mid 경계
- `HIGH_BOUNDARY`: 오차 ±5% 이하가 되는 최소 리뷰 수 → mid/high 경계

In [8]:
Z = 1.96

def wilson_margin(n):
    p = 0.5
    denom = 1 + Z**2 / n
    return (Z / denom) * np.sqrt(p*(1-p)/n + Z**2/(4*n**2)) * 100

def find_n_for_margin(target_pct):
    for n in range(1, 10000):
        if wilson_margin(n) <= target_pct:
            return n
    return 10000

LOW_BOUNDARY  = find_n_for_margin(15)
HIGH_BOUNDARY = find_n_for_margin(5)

print(f'low  (±15% 초과) : total_reviews <  {LOW_BOUNDARY}개')
print(f'mid  (±5~15%)    : {LOW_BOUNDARY} ≤ total_reviews < {HIGH_BOUNDARY}개')
print(f'high (±5% 이하)  : total_reviews >= {HIGH_BOUNDARY}개')

low  (±15% 초과) : total_reviews <  39개
mid  (±5~15%)    : 39 ≤ total_reviews < 381개
high (±5% 이하)  : total_reviews >= 381개


### 5-2. 층 할당

`primary_genre`와 리뷰 신뢰도를 조합하여 각 게임에 층을 배정한다.

In [9]:
def assign_trust(n):
    if n >= HIGH_BOUNDARY:
        return 'high'
    elif n >= LOW_BOUNDARY:
        return 'mid'
    else:
        return 'low'

df_filtered['trust']   = df_filtered['total_reviews'].apply(assign_trust)
df_filtered['stratum'] = df_filtered['primary_genre'] + '_' + df_filtered['trust']

pop = df_filtered['stratum'].value_counts().sort_index()
N   = len(df_filtered)

print(f'모집단: {N:,}개\n')
print(f'{"층":<22} {"게임 수":>8}  {"비중":>7}')
print('-' * 42)
for stratum, cnt in pop.items():
    print(f'{stratum:<22} {cnt:>8,}  {cnt/N*100:>6.1f}%')
print('-' * 42)
print(f'{"합계":<22} {N:>8,}  {100.0:>6.1f}%')

모집단: 8,730개

층                          게임 수       비중
------------------------------------------
Action_high                 270     3.1%
Action_low                1,189    13.6%
Action_mid                  725     8.3%
Adventure_high               88     1.0%
Adventure_low               360     4.1%
Adventure_mid               273     3.1%
Casual_high                  74     0.8%
Casual_low                  617     7.1%
Casual_mid                  342     3.9%
RPG_high                    262     3.0%
RPG_low                     555     6.4%
RPG_mid                     538     6.2%
Racing_high                  29     0.3%
Racing_low                  166     1.9%
Racing_mid                   88     1.0%
Simulation_high             163     1.9%
Simulation_low              501     5.7%
Simulation_mid              426     4.9%
Sports_high                  19     0.2%
Sports_low                  116     1.3%
Sports_mid                   98     1.1%
Strategy_high               362     4.1%
S

### 5-3. 표본 배분 — 비례 배분 + 최소 하한선

순수 비례 배분은 게임 수가 많은 장르에 표본이 과도하게 집중된다.
이를 보완하기 위해 **층당 최소 하한선**을 적용하고, 총합이 목표 수를 초과하면 큰 층에서 비례 축소한다.

| 파라미터 | 값 | 설명 |
|---|---|---|
| `TOTAL_N` | 200 | 목표 표본 수 |
| `MIN_PER_STRATUM` | 5 | 층당 최소 추출 수 |
| 층 수 | 24 | 8장르 × 3신뢰도 |

In [10]:
TOTAL_N         = 200
MIN_PER_STRATUM = 5
RANDOM_SEED     = 42

def compute_sample_plan(pop_series, total_n, min_floor):
    proportional = (pop_series / pop_series.sum() * total_n).round().astype(int)
    floored = proportional.clip(lower=min_floor)
    floored = floored.combine(pop_series, min)
    overflow = floored.sum() - total_n
    if overflow > 0:
        reducible = floored[(floored > min_floor) & (floored < pop_series)]
        if len(reducible) > 0:
            above = reducible - min_floor
            cut   = (above / above.sum() * overflow).round().astype(int)
            diff  = cut.sum() - overflow
            if diff != 0:
                cut.iloc[cut.argmax()] -= diff
            floored[reducible.index] -= cut
    return floored

sample_plan  = compute_sample_plan(pop, TOTAL_N, MIN_PER_STRATUM)
proportional = (pop / pop.sum() * TOTAL_N).round().astype(int)

print(f'목표 표본: {TOTAL_N}개  |  층당 최소: {MIN_PER_STRATUM}개\n')
print(f'{"층":<22} {"모집단":>8}  {"비중":>7}  {"비례":>6}  {"최종":>6}  {"추출률":>7}  비고')
print('-' * 76)
for stratum in pop.index:
    cnt  = pop[stratum]
    prop = proportional[stratum]
    n    = sample_plan[stratum]
    rate = n / cnt * 100
    note = '전수' if n == cnt else ('하한' if n == MIN_PER_STRATUM and prop < MIN_PER_STRATUM else '')
    print(f'{stratum:<22} {cnt:>8,}  {cnt/N*100:>6.1f}%  {prop:>6}  {n:>6}  {rate:>6.1f}%  {note}')
print('-' * 76)
print(f'{"합계":<22} {N:>8,}  {"100.0%":>7}  {proportional.sum():>6}  {sample_plan.sum():>6}')

SAMPLE_PLAN = sample_plan.to_dict()

목표 표본: 200개  |  층당 최소: 5개

층                           모집단       비중      비례      최종      추출률  비고
----------------------------------------------------------------------------
Action_high                 270     3.1%       6       6     2.2%  
Action_low                1,189    13.6%      27      22     1.9%  
Action_mid                  725     8.3%      17      14     1.9%  
Adventure_high               88     1.0%       2       5     5.7%  하한
Adventure_low               360     4.1%       8       7     1.9%  
Adventure_mid               273     3.1%       6       6     2.2%  
Casual_high                  74     0.8%       2       5     6.8%  하한
Casual_low                  617     7.1%      14      12     1.9%  
Casual_mid                  342     3.9%       8       7     2.0%  
RPG_high                    262     3.0%       6       6     2.3%  
RPG_low                     555     6.4%      13      11     2.0%  
RPG_mid                     538     6.2%      12      10     1.9%  
Racing

### 5-4. 층화 추출

In [11]:
import pandas as pd

sampled_frames = []
for stratum, n in SAMPLE_PLAN.items():
    pool     = df_filtered[df_filtered['stratum'] == stratum]
    actual_n = min(n, len(pool))
    sample   = pool.sample(n=actual_n, random_state=RANDOM_SEED)
    sampled_frames.append(sample)

df_sample = pd.concat(sampled_frames).reset_index(drop=True)
print(f'추출 완료: {len(df_sample)}개')
print(df_sample['stratum'].value_counts().sort_index())

추출 완료: 200개
stratum
Action_high         6
Action_low         22
Action_mid         14
Adventure_high      5
Adventure_low       7
Adventure_mid       6
Casual_high         5
Casual_low         12
Casual_mid          7
RPG_high            6
RPG_low            11
RPG_mid            10
Racing_high         5
Racing_low          5
Racing_mid          5
Simulation_high     5
Simulation_low     10
Simulation_mid      9
Sports_high         5
Sports_low          5
Sports_mid          5
Strategy_high       7
Strategy_low       15
Strategy_mid       13
Name: count, dtype: int64


### 5-5. 검증

층별 모집단 비중과 표본 비중을 비교하여 대표성을 확인한다.

In [12]:
pop_ratio  = df_filtered['stratum'].value_counts(normalize=True).sort_index() * 100
samp_ratio = df_sample['stratum'].value_counts(normalize=True).sort_index() * 100
samp_count = df_sample['stratum'].value_counts().sort_index()

print(f'=== 층별 모집단 vs 표본 비교 ===')
print(f'{"층":<22} {"모집단%":>8}  {"표본%":>7}  {"격차":>7}  {"표본n":>6}')
print('-' * 58)
for stratum in pop_ratio.index:
    p   = pop_ratio[stratum]
    s   = samp_ratio.get(stratum, 0)
    n   = samp_count.get(stratum, 0)
    gap = s - p
    flag = ' ⚠' if abs(gap) > 10 else ''
    print(f'{stratum:<22} {p:>7.1f}%  {s:>6.1f}%  {gap:>+6.1f}%  {n:>6}{flag}')
print('-' * 58)

print(f'\n=== 체크리스트 ===')
checks = [
    ('층당 최소 하한 충족', all(samp_count >= MIN_PER_STRATUM)),
    ('총 표본 수',         len(df_sample) == TOTAL_N),
]
for label, ok in checks:
    print(f'  [{"✓" if ok else "✗"}] {label}')

=== 층별 모집단 vs 표본 비교 ===
층                          모집단%      표본%       격차     표본n
----------------------------------------------------------
Action_high                3.1%     3.0%    -0.1%       6
Action_low                13.6%    11.0%    -2.6%      22
Action_mid                 8.3%     7.0%    -1.3%      14
Adventure_high             1.0%     2.5%    +1.5%       5
Adventure_low              4.1%     3.5%    -0.6%       7
Adventure_mid              3.1%     3.0%    -0.1%       6
Casual_high                0.8%     2.5%    +1.7%       5
Casual_low                 7.1%     6.0%    -1.1%      12
Casual_mid                 3.9%     3.5%    -0.4%       7
RPG_high                   3.0%     3.0%    -0.0%       6
RPG_low                    6.4%     5.5%    -0.9%      11
RPG_mid                    6.2%     5.0%    -1.2%      10
Racing_high                0.3%     2.5%    +2.2%       5
Racing_low                 1.9%     2.5%    +0.6%       5
Racing_mid                 1.0%     2.5%    +1.

### 5-6. 표본 저장

층화 추출 결과를 CSV로 저장한다. 리뷰 수집 스크립트의 입력 파일로 사용된다.

In [13]:
OUT_COLS = [
    'appid', 'name', 'release_date', 'genres',
    'positive', 'negative', 'total_reviews', 'positive_rate', 'price',
    'developers', 'primary_genre', 'trust', 'stratum',
]

out_path = '../../data/preprocessed/steam_indie_genre_stratified_sample.csv'
df_sample[OUT_COLS].to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_sample)}개)')
df_sample[OUT_COLS].head()
df_sample[OUT_COLS].shape

저장 완료 → ../../data/preprocessed/steam_indie_genre_stratified_sample.csv (200개)


(200, 13)

## 6. 표본 장르 분포 확인

추출된 200개 표본의 장르 × 리뷰 신뢰도 분포를 시각화하여 층화 결과를 검증한다.

In [14]:
# ── 1. 장르별 표본 수 (bar chart) ──────────────────────────────────────────
genre_dist = df_sample['primary_genre'].value_counts().reset_index()
genre_dist.columns = ['genre', 'count']
pop_genre  = df_filtered['primary_genre'].value_counts().reset_index()
pop_genre.columns = ['genre', 'pop_count']
genre_dist = genre_dist.merge(pop_genre, on='genre')
genre_dist['sample_ratio'] = genre_dist['count'] / genre_dist['count'].sum() * 100
genre_dist['pop_ratio']    = genre_dist['pop_count'] / genre_dist['pop_count'].sum() * 100

fig = go.Figure()
fig.add_trace(go.Bar(
    x=genre_dist['genre'], y=genre_dist['pop_ratio'],
    name='모집단 비중(%)', marker_color='lightsteelblue', opacity=0.7
))
fig.add_trace(go.Bar(
    x=genre_dist['genre'], y=genre_dist['sample_ratio'],
    name='표본 비중(%)', marker_color='steelblue',
    text=genre_dist['count'].apply(lambda x: f'{x}개'),
    textposition='outside'
))
fig.update_layout(
    title='장르별 모집단 vs 표본 비중 비교',
    xaxis_title='장르', yaxis_title='비중 (%)',
    barmode='group', height=450,
    xaxis_categoryorder='total descending'
)
fig.show()

**해석:** 장르별 추출 표본 수를 확인한다. Action·Adventure·Casual 등 게임 수가 많은 장르에서 표본이 더 많이 배정되며, 비례 배분 원칙에 따라 시장 규모를 반영한 대표성이 확보됐다.

In [15]:
# ── 2. 장르 × 신뢰도 히트맵 ────────────────────────────────────────────────
heatmap_data = (
    df_sample.groupby(['primary_genre', 'trust'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['high', 'mid', 'low'], fill_value=0)
)

fig2 = go.Figure(go.Heatmap(
    z=heatmap_data.values,
    x=['high', 'mid', 'low'],
    y=heatmap_data.index.tolist(),
    text=heatmap_data.values,
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=True,
))
fig2.update_layout(
    title='장르 × 리뷰 신뢰도 표본 수 히트맵',
    xaxis_title='리뷰 신뢰도',
    yaxis_title='장르',
    height=450
)
fig2.show()

**해석:** 8개 장르와 3단계 신뢰도(high·mid·low) 교차 조합별 표본 분포를 확인한다. 각 층에서 최소 5개 표본이 확보됐으며, 전체 200개 표본이 장르·신뢰도 구조를 고르게 반영하도록 배분됐다.

In [16]:
# ── 3. 신뢰도별 표본 수 (pie chart) ────────────────────────────────────────
trust_dist = df_sample['trust'].value_counts().reindex(['high', 'mid', 'low'])

fig3 = go.Figure(go.Pie(
    labels=trust_dist.index,
    values=trust_dist.values,
    hole=0.4,
    marker_colors=['#2196F3', '#90CAF9', '#E3F2FD'],
    textinfo='label+percent+value'
))
fig3.update_layout(
    title='리뷰 신뢰도별 표본 비중',
    height=400
)
fig3.show()